```
┌─────────────────────────────────────────────────────────────────┐
│                    MODELING PIPELINE                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Raw Audio ──→ Preprocessing ──→ Mel Spectrogram ──→ CNN/Transformer
│                     │                                    │      │
│                     ├── Augmentation                     │      │
│                     ├── Quality Filter                   │      │
│                     └── Chunk Splitting                  │      │
│                                                          │      │
│                              ┌────────────────────────────┘     │
│                              ▼                                  │
│                    ┌──── Baseline (EfficientNet-B0) ────┐       │
│                    ├──── Mid-tier (EfficientNet-B3/V2) ─┤       │
│                    ├──── Advanced (BEATs / AST) ────────┤       │
│                    └──── Ensemble + TTA ────────────────┘       │
│                              │                                  │
│                              ▼                                  │
│                    Post-processing + Submission                 │
└─────────────────────────────────────────────────────────────────┘
```

In [11]:
# =============================================================================
# Cell 1: Imports and Configuration
# =============================================================================
"""
This cell serves as the SINGLE SOURCE OF TRUTH for the entire notebook.
- All imports are grouped and documented
- All hyperparameters live in the CFG class (easy to experiment with)
- Reproducibility is enforced via seed_everything()
"""
import os
import gc
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
# ------------------------------ DATA & VISUALIZATION ------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# ------------------------------ AUDIO PROCESSING ------------------------------
import librosa
import soundfile as sf
# ------------------------------ DEEP LEARNING ------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast  # Mixed precision training
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, OneCycleLR
# ------------------------------ EXTERNAL LIBRARIES ------------------------------
import timm  # Pretrained model zoo (EfficientNetV2, etc.)
import albumentations as A  # Fast image augmentations (applied to spectrograms)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, f1_score
from tqdm.notebook import tqdm  # Progress bars for notebooks

import colorednoise as cn  # Colored noise augmentation (pip install colorednoise)

warnings.filterwarnings('ignore')  # Cleaner notebook output


# =============================================================================
# Configuration - Single source of truth
# =============================================================================

class CFG:
    """
    Centralized configuration class.
    
    All paths, audio parameters, training hyperparameters, model settings,
    and augmentation parameters are defined here.
    """
    # ========================== PATHS ==========================
    PROJECT_ROOT = Path.cwd().resolve()
    BASE_PATH = PROJECT_ROOT / "kaggle/input/competitions/birdclef-2026"
    TRAIN_CSV = BASE_PATH / "train.csv"
    AUDIO_DIR = BASE_PATH / "train_audio"
    OUTPUT_DIR = Path("/kaggle/working")  # Kaggle working directory

    # ========================== AUDIO SETTINGS ==========================
    SR = 32000  # Sample rate
    DURATION = 5  # Duration of each audio chunk in seconds
    N_SAMPLES = SR * DURATION  # 160000 samples per chunk

    # ========================== MEL SPECTROGRAM ==========================
    # These parameters convert raw audio into a 2D image (mel spectrogram)
    N_MELS = 128  # Number of mel frequency bins (height of image)
    N_FFT = 2048  # FFT window size
    HOP_LENGTH = 512  # Hop length between frames (width of image)
    FMIN = 50  # Minimum frequency (Hz)
    FMAX = 14000  # Maximum frequency (Hz) - most species vocalizations
    POWER = 2.0  # Power for mel spectrogram (2.0 = power, 1.0 = magnitude)
    TOP_DB = 80  # Dynamic range for amplitude-to-dB conversion

    # ========================== TRAINING SETTINGS ==========================
    SEED = 42  # Master random seed for full reproducibility
    N_FOLDS = 5  # Number of folds for Stratified K-Fold CV
    TRAIN_FOLDS = [0, 1, 2, 3]  # Which folds to actually train on (useful for quick tests)
    EPOCHS = 30  # Total training epochs
    BATCH_SIZE = 32  # Batch size (adjust based on GPU memory)
    LR = 1e-3  # Initial learning rate
    MIN_LR = 1e-6  # Minimum learning rate (for schedulers)
    WEIGHT_DECAY = 1e-4  # L2 regularization strength
    WARMUP_EPOCHS = 2  # Number of warmup epochs

    # ========================== MODEL SETTINGS ==========================
    MODEL_NAME = "tf_efficientnetv2_s"  # timm model - excellent speed/accuracy balance
    PRETRAINED = True  # Use ImageNet pretrained weights
    NUM_CLASSES = 206  # Number of species (update after EDA)
    IN_CHANNELS = 1  # 1 = grayscale spectrogram input

    # ========================== AUGMENTATION ==========================
    MIXUP_ALPHA = 0.5  # MixUp alpha parameter
    CUTMIX_ALPHA = 1.0  # CutMix alpha parameter
    MIXUP_PROB = 0.5  # Probability of applying MixUp/CutMix

    # ========================== DATA QUALITY FILTER ==========================
    MIN_RATING = 2.0  # Minimum rating to include training samples

    # ========================== HARDWARE & PERFORMANCE ==========================
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 4  # DataLoader workers
    PIN_MEMORY = True  # Faster data transfer to GPU
    USE_AMP = True  # Automatic Mixed Precision (faster + less VRAM)

    # ========================== INFERENCE ==========================
    TTA_STEPS = 5  # Test-Time Augmentation steps


def seed_everything(seed=42):
    """
    Set random seeds across all libraries for full reproducibility.
    
    Args:
        seed (int): Random seed value. Default is 42.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make cuDNN deterministic (slower but 100% reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# Initialize everything
# =============================================================================
seed_everything(CFG.SEED)
print(f"🖥️ Device: {CFG.DEVICE}")
print(f"📊 Config loaded:")
print(f"   • Model: {CFG.MODEL_NAME}")
print(f"   • Audio: {CFG.DURATION}s @ {CFG.SR}Hz → {CFG.N_SAMPLES} samples")
print(f"   • Training folds: {CFG.TRAIN_FOLDS}")
print(f"   • Mixed Precision (AMP): {CFG.USE_AMP}")

🖥️ Device: cpu
📊 Config loaded:
   • Model: tf_efficientnetv2_s
   • Audio: 5s @ 32000Hz → 160000 samples
   • Training folds: [0, 1, 2, 3]
   • Mixed Precision (AMP): True


In [12]:
# =============================================================================
# Cell 2: Data Preparation & Fold Splitting
# =============================================================================
"""
This cell performs the critical data preparation steps:
1. Loads the training CSV
2. Creates label encoders (string ↔ integer)
3. Filters low-quality recordings
4. Creates Stratified K-Fold splits
5. Computes class weights to handle severe class imbalance

This cell must be run after Cell 1 (CFG is used extensively here).
"""

# =============================================================================
# 1. Load Training Data
# =============================================================================
df = pd.read_csv(CFG.TRAIN_CSV)
print(f"📥 Loaded training data: {len(df):,} rows × {len(df.columns)} columns")

# =============================================================================
# 2. Build Label Encoder
# =============================================================================
# We convert string labels (e.g. "amegfi") into integer indices (0, 1, 2, ...)
# This is required for PyTorch's CrossEntropyLoss
labels_sorted = sorted(df['primary_label'].unique())
label2idx = {label: idx for idx, label in enumerate(labels_sorted)}
idx2label = {idx: label for label, idx in label2idx.items()}

# Update global config with actual number of classes
CFG.NUM_CLASSES = len(labels_sorted)
print(f"🏷️  Found {CFG.NUM_CLASSES} unique bird species")

# =============================================================================
# 3. Add Useful Columns
# =============================================================================
# label_idx   → integer target for training
# filepath    → full path to the audio file (makes Dataset class cleaner)
df['label_idx'] = df['primary_label'].map(label2idx)
df['filepath'] = df['filename'].apply(lambda x: str(CFG.AUDIO_DIR / x))

# =============================================================================
# 4. Quality Filter (Optional but Recommended)
# =============================================================================
print(f"Before quality filter: {len(df)}")

if 'rating' in df.columns:
    # Keep only recordings with rating >= MIN_RATING (usually 2.0 or 3.0)
    df_filtered = df[df['rating'] >= CFG.MIN_RATING].reset_index(drop=True)
    print(f"After quality filter (rating >= {CFG.MIN_RATING}): {len(df_filtered):,} "
          f"samples" f"({len(df_filtered) / len(df) * 100:.1f}% kept)")
else:
    print("⚠️  No 'rating' column found — skipping quality filter")
    df_filtered = df.copy()

# =============================================================================
# 5. Stratified K-Fold Cross-Validation
# =============================================================================
# We use StratifiedKFold to ensure each fold has roughly the same class distribution.
# This is very important for imbalanced datasets like BirdCLEF. 
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
df_filtered['fold'] = -1  # placeholder

# Assign fold number to each row
# Note: We only assign to val_idx because each row appears in validation set exactly once
for fold, (train_idx, val_idx) in enumerate(skf.split(df_filtered, df_filtered['label_idx'])):
    df_filtered.loc[val_idx, 'fold'] = fold

print(f"\n📊 Classes: {CFG.NUM_CLASSES}")
print(f"📁 Fold distribution:\n{df_filtered['fold'].value_counts().sort_index()}")

# =============================================================================
# 6. Compute Class Weights (Handle Severe Imbalance)
# =============================================================================
# BirdCLEF has extremely imbalanced classes (some species have < 10 recordings,
# others have > 500). We use inverse frequency weighting.
class_counts = df_filtered['label_idx'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts + 1)  # inverse frequency
class_weights = class_weights / class_weights.sum() * CFG.NUM_CLASSES  # normalize
class_weights_tensor = torch.FloatTensor(class_weights).to(CFG.DEVICE)

print(f"\n⚖️  Class weight statistics:")
print(f"   • Min weight : {class_weights.min():.4f}")
print(f"   • Max weight : {class_weights.max():.4f}")
print(f"   • Mean weight: {class_weights.mean():.4f}")
print(f"   • Tensor shape: {class_weights_tensor.shape} → ready for loss function")

📥 Loaded training data: 35,549 rows × 15 columns
🏷️  Found 206 unique bird species
Before quality filter: 35549
After quality filter (rating >= 2.0): 22,411 samples(63.0% kept)

📊 Classes: 206
📁 Fold distribution:
fold
0    4483
1    4482
2    4482
3    4482
4    4482
Name: count, dtype: int64

⚖️  Class weight statistics:
   • Min weight : 0.0498
   • Max weight : 10.3088
   • Mean weight: 1.1135
   • Tensor shape: torch.Size([185]) → ready for loss function


In [15]:
# =============================================================================
# Cell 3: Audio Augmentations (Time-Domain)
# =============================================================================
"""
This cell defines two augmentation classes used during training:

1. AudioAugmentations  → Time-domain augmentations (applied to raw waveform)
2. SpecAugmentations   → Frequency-domain augmentations (applied to mel spectrogram)

These augmentations help the model generalize better to:
- Background noise (Gaussian, pink)
- Recording variations (gain, fade, time shift)
- Bird vocalization variations (pitch shift, time stretch)
- Spectrogram occlusions (SpecAugment-style masking)

All methods are static for easy use inside the Dataset class.
"""


# =============================================================================
# 1. Time-Domain Audio Augmentations
# =============================================================================
class AudioAugmentations:
    """
    Efficient time-domain audio augmentations.
    
    These are applied to the raw waveform BEFORE converting to mel spectrogram.
    They simulate real-world recording variations and help the model become
    robust to noise, volume changes, and slight timing differences.
    """

    @staticmethod
    def add_gaussian_noise(audio: np.ndarray, min_snr_db: float = 5, max_snr_db: float = 20) -> np.ndarray:
        """
        Add white Gaussian noise with random Signal-to-Noise Ratio (SNR).
        
        Args:
            audio: Input waveform (1D numpy array)
            min_snr_db: Minimum SNR in dB (higher = less noise)
            max_snr_db: Maximum SNR in dB
            
        Returns:
            Noisy waveform (float32)
        """
        snr_db = np.random.uniform(min_snr_db, max_snr_db)
        signal_power = np.mean(audio ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = np.random.normal(0, np.sqrt(noise_power), len(audio))
        return audio + noise.astype(np.float32)

    @staticmethod
    def add_pink_noise(audio, min_snr_db=10, max_snr_db=30):
        """
        Add pink noise (1/f noise) - more realistic for natural environments.
        
        Pink noise has equal power per octave (more low-frequency energy).
        This is very common in outdoor bird recordings (wind, distant traffic, etc.).
        """
        snr_db = np.random.uniform(min_snr_db, max_snr_db)
        signal_power = np.mean(audio ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = cn.powerlaw_psd_gaussian(1, len(audio))  # beta=1 for pink
        noise = noise * np.sqrt(noise_power) / (np.std(noise) + 1e-8)
        return audio + noise.astype(np.float32)

    @staticmethod
    def time_shift(audio: np.ndarray, max_shift_pct: float = 0.3) -> np.ndarray:
        """
        Randomly shift the audio in time (circular shift).
        
        Simulates birds singing at slightly different positions in the 5-second window.
        """
        shift = int(len(audio) * np.random.uniform(-max_shift_pct, max_shift_pct))
        return np.roll(audio, shift)

    @staticmethod
    def time_stretch(audio: np.ndarray, rate_range: tuple = (0.8, 1.2)) -> np.ndarray:
        """
        Randomly stretch or compress audio in time (change speed without changing pitch).
        
        Helps the model become invariant to singing speed variations between individuals.
        """
        rate = np.random.uniform(*rate_range)
        stretched = librosa.effects.time_stretch(audio, rate=rate)
        # Resize to original length
        if len(stretched) > len(audio):
            stretched = stretched[:len(audio)]
        else:
            stretched = np.pad(stretched, (0, len(audio) - len(stretched)))
        return stretched

    @staticmethod
    def pitch_shift(audio: np.ndarray, sr: int, n_steps_range: tuple = (-2, 2)) -> np.ndarray:
        """
        Randomly shift pitch up or down (in semitones).
        
        Simulates different bird individuals, ages, or recording equipment variations.
        """
        n_steps = np.random.uniform(*n_steps_range)
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

    @staticmethod
    def random_gain(audio: np.ndarray, min_gain_db: float = -6, max_gain_db: float = 6) -> np.ndarray:
        """
        Random volume change (in dB).
        
        Makes the model robust to different recording volumes and distances from the bird.
        """
        gain_db = np.random.uniform(min_gain_db, max_gain_db)
        gain = 10 ** (gain_db / 20)
        return audio * gain

    @staticmethod
    def fade(audio: np.ndarray, fade_in_pct: float = 0.01, fade_out_pct: float = 0.01) -> np.ndarray:
        """
        Apply linear fade-in and fade-out.
        
        Reduces abrupt starts/ends that can occur when cutting 5-second chunks.
        """
        fade_in = int(len(audio) * fade_in_pct)
        fade_out = int(len(audio) * fade_out_pct)
        audio = audio.copy()
        audio[:fade_in] *= np.linspace(0, 1, fade_in)
        audio[-fade_out:] *= np.linspace(1, 0, fade_out)
        return audio


# =============================================================================
# Spectrogram Augmentations (Frequency-Domain)
# =============================================================================

class SpecAugmentations:
    """
    Spectrogram-level augmentations inspired by SpecAugment (Google, 2019).
    
    These are applied AFTER converting audio to mel spectrogram.
    They force the model to learn from partial information (occluded frequencies or time steps).
    """

    @staticmethod
    def _random_mask(spec: torch.Tensor, dim: int, max_mask_pct: float, num_masks: int) -> torch.Tensor:
        """
        Internal helper: randomly masks contiguous regions along a given dimension.
        
        Args:
            spec: Input spectrogram tensor
            dim: Dimension to mask (-2 = frequency, -1 = time)
            max_mask_pct: Maximum percentage of the dimension to mask (0.0 - 1.0)
            num_masks: Number of separate masks to apply
            
        Returns:
            Masked spectrogram (cloned, not in-place)
        """
        spec = spec.clone()
        dim_size = spec.shape[dim]

        for _ in range(num_masks):
            mask_size = int(dim_size * np.random.uniform(0, max_mask_pct))
            start = np.random.randint(0, max(1, dim_size - mask_size))

            if dim == -2:  # Frequency mask (vertical)
                spec[..., start:start + mask_size, :] = 0
            else:  # Time mask (horizontal)
                spec[..., :, start:start + mask_size] = 0

        return spec

    @staticmethod
    def freq_mask(spec: torch.Tensor, max_mask_pct: float = 0.15, num_masks: int = 2):
        """
        Randomly mask contiguous frequency bands (vertical masks).
        
        Simulates missing frequency information (e.g., due to microphone limitations
        or environmental filtering).
        """
        return SpecAugmentations._random_mask(spec, dim=-2,
                                              max_mask_pct=max_mask_pct,
                                              num_masks=num_masks)

    @staticmethod
    def time_mask(spec: torch.Tensor, max_mask_pct: float = 0.15, num_masks: int = 2):
        """
        Randomly mask contiguous time steps (horizontal masks).
        
        Forces the model to recognize birds even when parts of the call are occluded
        (e.g., overlapping sounds, wind gusts).
        """
        return SpecAugmentations._random_mask(spec, dim=-1,
                                              max_mask_pct=max_mask_pct,
                                              num_masks=num_masks)

    @staticmethod
    def mixup(spec1, label1, spec2, label2: torch.Tensor, alpha: float = 0.5):
        """
        MixUp augmentation on spectrograms + soft labels.
        
        Creates convex combinations of two examples. Improves generalization
        and calibration. Returns mixed spectrogram and mixed (soft) label.
        """
        lam = np.random.beta(alpha, alpha)
        mixed_spec = lam * spec1 + (1 - lam) * spec2
        mixed_label = lam * label1 + (1 - lam) * label2
        return mixed_spec, mixed_label


print("✅ Augmentation modules ready.")

✅ Augmentation modules ready.


In [18]:
# =============================================================================
# Cell 4: Dataset Class
# =============================================================================
"""
    This cell defines the core `BirdCLEFDataset` class that handles the entire
    data pipeline for each sample:
    
    1. On-the-fly audio loading (memory efficient — no preloading)
    2. Smart duration handling (tile short clips, random/center crop long clips)
    3. Probabilistic time-domain augmentations
    4. Fast mel spectrogram conversion with precomputed filter bank
    5. Spectrogram augmentations (FreqMask + TimeMask)
    6. One-hot label encoding (ready for MixUp)
    """


class BirdCLEFDataset(Dataset):
    """
    Optimized PyTorch Dataset for BirdCLEF 2026.
    
    Key design choices:
    - Loads audio on-the-fly (very memory efficient)
    - Uses precomputed mel filter bank for ~30% faster spectrogram generation
    - Applies different cropping strategies for train vs validation
    - Probabilistic augmentations (easy to tune)
    - Returns one-hot labels for MixUp compatibility
    """

    def __init__(self, df, cfg, mode: str = 'train', augment_audio: bool = True, augment_spec: bool = True):
        """
        Args:
            df: DataFrame with columns ['filepath', 'label_idx', ...]
            cfg: Configuration object (CFG class)
            mode: 'train' or 'val' (affects cropping and augmentation)
            augment_audio: Whether to apply time-domain augmentations
            augment_spec: Whether to apply spectrogram augmentations
        """
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.mode = mode
        self.augment_audio = augment_audio and (mode == 'train')
        self.augment_spec = augment_spec and (mode == 'train')
        self.audio_aug = AudioAugmentations()
        self.spec_aug = SpecAugmentations()

        # Precompute mel filter bank once (big speed boost)
        self.mel_basis = librosa.filters.mel(
            sr=cfg.SR,
            n_fft=cfg.N_FFT,
            n_mels=cfg.N_MELS,
            fmin=cfg.FMIN,
            fmax=cfg.FMAX
        )

    def __len__(self):
        return len(self.df)

    def _load_audio(self, filepath: str) -> np.ndarray:
        """
        Load audio and ensure exactly N_SAMPLES length.
        
        Strategy:
        - Short audio → tile (repeat) instead of zero-pad (better for training)
        - Long audio  → random crop (train) or center crop (val)
        """
        try:
            audio, sr = librosa.load(filepath, sr=self.cfg.SR, mono=True)
        except Exception:
            # Fallback: silent audio
            return np.zeros(self.cfg.N_SAMPLES, dtype=np.float32)

        # Handle duration
        target_len = self.cfg.N_SAMPLES

        if len(audio) < target_len:
            # Tile (repeat) short audio — preserves energy better than zero-padding
            repeats = math.ceil(target_len / len(audio))
            audio = np.tile(audio, repeats)[:target_len]
        elif len(audio) > target_len:
            if self.mode == 'train':
                # Random crop during training → more variety
                max_start = len(audio) - target_len
                start = np.random.randint(0, max_start)
                audio = audio[start:start + target_len]
            else:
                # Center crop during validation → deterministic
                start = (len(audio) - target_len) // 2
                audio = audio[start:start + target_len]

        return audio.astype(np.float32)

    def _audio_to_melspec(self, audio: np.ndarray) -> np.ndarray:
        """
        Convert audio waveform to normalized log-mel spectrogram.
        
        Output shape: (n_mels, time_steps)
        Values are normalized to [0, 1] per sample.
        """
        S = librosa.feature.melspectrogram(
            y=audio,
            sr=self.cfg.SR,
            n_mels=self.cfg.N_MELS,
            n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH,
            fmin=self.cfg.FMIN,
            fmax=self.cfg.FMAX,
            power=self.cfg.POWER,
        )
        # Convert to dB and normalize per sample
        S_db = librosa.power_to_db(S, ref=np.max, top_db=self.cfg.TOP_DB)
        S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)

        return S_db.astype(np.float32)

    def _apply_audio_augmentations(self, audio: np.ndarray) -> np.ndarray:
        """
        Apply random time-domain augmentations with tuned probabilities.
        
        Probabilities are intentionally conservative to avoid destroying
        the bird vocalization signal.
        """
        if np.random.random() < 0.5:
            audio = self.audio_aug.add_gaussian_noise(audio)
        if np.random.random() < 0.3:
            audio = self.audio_aug.add_pink_noise(audio)
        if np.random.random() < 0.3:
            audio = self.audio_aug.time_shift(audio)
        if np.random.random() < 0.2:
            audio = self.audio_aug.random_gain(audio)
        if np.random.random() < 0.1:
            audio = self.audio_aug.pitch_shift(audio, self.cfg.SR)
        return audio

    def __getitem__(self, idx):
        """
        Returns:
            melspec: torch.Tensor of shape (1, n_mels, time_steps)
            label:   torch.Tensor of shape (NUM_CLASSES,) — one-hot encoded
        """
        row = self.df.iloc[idx]
        filepath = row['filepath']
        label_idx = row['label_idx']

        # 1. Load + preprocess audio
        audio = self._load_audio(filepath)

        # 2. Apply time-domain augmentations
        if self.augment_audio:
            audio = self._apply_audio_augmentations(audio)

        # 3. Convert to mel spectrogram
        melspec = self._audio_to_melspec(audio)

        # 4. To tensor: (1, n_mels, time_steps) — ready for Conv2d
        melspec = torch.from_numpy(melspec).unsqueeze(0)

        # 5. Apply spectrogram augmentations
        if self.augment_spec:
            if np.random.random() < 0.5:  # Apply Freq Mask with probability
                melspec = self.spec_aug.freq_mask(melspec)
            if np.random.random() < 0.5:  # Apply Time Mask with probability
                melspec = self.spec_aug.time_mask(melspec)

        # One-hot label for mixup compatibility
        label = torch.zeros(self.cfg.NUM_CLASSES, dtype=torch.float32)
        label[label_idx] = 1.0

        return melspec, label


# =============================================================================
# Quick Test
# =============================================================================
print("\n" + "=" * 60)
print("SANITY CHECK: Testing BirdCLEFDataset class")
print("=" * 60)

# Split data using fold 0 as validation (standard 5-fold setup)
train_df = df_filtered[df_filtered['fold'] != 0].reset_index(drop=True)
val_df = df_filtered[df_filtered['fold'] == 0].reset_index(drop=True)

print(f"Training samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")

# Create tiny test dataset with only 5 samples (for speed)
test_ds = BirdCLEFDataset(train_df.head(5), CFG, mode='train')
# Fetch first sample — this runs the entire pipeline
sample_spec, sample_label = test_ds[0]
print(f"✅ Dataset test passed.")
print(f"   Spectrogram shape: {sample_spec.shape}")
print(f"   Label shape: {sample_label.shape}")
print(f"   Label sum: {sample_label.sum()}")
print(f"   Active class idx  : {sample_label.argmax().item()}")
print("=" * 60 + "\n")


SANITY CHECK: Testing BirdCLEFDataset class
Training samples: 17,928
Validation samples: 4,483
✅ Dataset test passed.
   Spectrogram shape: torch.Size([1, 128, 313])
   Label shape: torch.Size([206])
   Label sum: 1.0
   Active class idx  : 5



In [20]:
# =============================================================================
# Cell 5: Model Architectures
# =============================================================================
"""
BirdCLEF 2026 - Cell 5: Custom Model Architectures

This cell defines the complete model architecture used for sound classification:

1. GeMPooling       → Generalized Mean Pooling (better than avg pooling for fine-grained audio)
2. AttentionHead    → Attention-based classification head (aggregates time-frequency features)
3. BirdCLEFModel    → Main model combining a pretrained CNN backbone + dual prediction heads

Input：el spectrograms (batch, 1, 128, 313)
          ↓
     Pretrained CNN Backbone (EfficientNetV2 ...)
          ↓
     Featured charts (batch, 1280, 4, 10)   ← the numbers depend on models
          ↓
     ┌──────────────────────┬──────────────────────┐
     ↓                      ↓                      ↓
  GeM Pooling          Attention Head         （two calculation in parallell）
     ↓                      ↓
  logits_gem         logits_attn
     ↓                      ↓
     └──────────┬───────────┘
                ↓
          output = 0.5 * logits_gem + 0.5 * logits_attn

Key design choices:
- Uses timm pretrained models (EfficientNetV2, ConvNeXt, etc.) adapted for 1-channel mel spectrograms
- Dual-path architecture: GeM pooling + Attention (both contribute to final prediction)
- Frame-level attention enables the model to focus on important time-frequency regions
- Supports both clip-level and frame-level predictions
"""


# =============================================================================
# 1. Generalized Mean Pooling (GeM)
# =============================================================================
class GeMPooling(nn.Module):
    """
    Generalized Mean Pooling layer.
    
    Unlike standard average pooling, GeM uses a learnable parameter 'p' that
    allows the model to learn the optimal pooling behavior between:
    - p=1 → Average pooling
    - p→∞ → Max pooling
    
    This is especially effective for fine-grained audio tasks like bird species
    identification, where certain frequency bands are more discriminative.
    """

    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)  # Learnable parameter
        self.eps = eps  # Small constant for numerical stability

    def forward(self, x) -> torch.Tensor:
        """
        Args:
            x: Feature map of shape (batch, channels, height, width)
        Returns:
            Pooled tensor of shape (batch, channels, 1, 1)
        """
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), 1
        ).pow(1.0 / self.p)


# =============================================================================
# 2. Attention-based Classification Head
# =============================================================================
class AttentionHead(nn.Module):
    """
    Attention mechanism for aggregating variable-length feature sequences.
    
    Instead of simple averaging or max-pooling, this head learns to assign
    importance weights to different time-frequency regions of the spectrogram.
    This helps the model focus on the most discriminative parts of the bird call.
    """

    def __init__(self, in_features, num_classes, hidden_dim=512):
        super().__init__()
        # Attention scoring network
        self.attention = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        """
        Args:
            x: Feature sequence of shape (batch, time_steps, features)
        Returns:
            logits: (batch, num_classes)
        """
        attn_weights = F.softmax(self.attention(x), dim=1)  # (batch, time, 1)
        context = (attn_weights * x).sum(dim=1)  # (batch, features)
        return self.classifier(context)


# =============================================================================
# 3. Main BirdCLEF Model
# =============================================================================
class BirdCLEFModel(nn.Module):
    """
    Main model for BirdCLEF 2026.
    
    Architecture:
    - Pretrained CNN backbone (EfficientNetV2 / ConvNeXt / etc.) from timm
    - Input adapted to single-channel mel spectrograms
    - Dual prediction heads:
        1. GeM pooling + MLP head (global context)
        2. Attention head (local, time-frequency aware)
    - Final prediction = 0.5 * GeM logits + 0.5 * Attention logits
    
    This dual-path design combines the strengths of both global and local feature aggregation.
    """

    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.cfg = cfg

        # ------------------------------------------------------------------
        # Backbone (pretrained CNN)
        # ------------------------------------------------------------------
        self.backbone = timm.create_model(
            cfg.MODEL_NAME,
            pretrained=pretrained,
            in_chans=cfg.IN_CHANNELS,  # 1 = single-channel mel spectrogram
            num_classes=0,  # Remove original classifier, (we add our own)
            global_pool='',  # Remove global pooling (we add our own GeM)
        )

        # Determine feature dimension dynamically
        with torch.no_grad():  # Do not calculate gradients
            dummy = torch.randn(1, cfg.IN_CHANNELS, cfg.N_MELS, 313)  # ~5s at hop=512
            features = self.backbone(dummy)
            self.feature_dim = features.shape[
                1]  # record number of channels, will be used to get num_classes' value after
            print(f"   Backbone feature dim: {self.feature_dim}")
            print(f"   Feature map shape: {features.shape}")

        # ------------------------------------------------------------------
        # Pooling + Heads
        # ------------------------------------------------------------------
        self.gem_pool = GeMPooling(p=3)

        self.head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, cfg.NUM_CLASSES),
        )

        # Optional: frame-level attention head
        self.use_attention = True
        if self.use_attention:
            self.attention_head = AttentionHead(
                self.feature_dim, cfg.NUM_CLASSES, hidden_dim=512
            )

    def forward(self, x) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            x: Input mel spectrogram of shape (batch, 1, n_mels, time_steps)
        Returns:
            logits: Combined predictions of shape (batch, num_classes)
        """
        # Extract features from backbone
        features = self.backbone(x)  # (batch, C, H, W)

        # Path 1: GeM pooling
        pooled = self.gem_pool(features).squeeze(-1).squeeze(-1)  # only (batch, C)
        logits_gem = self.head(pooled)

        if self.use_attention:
            # Path 2: Attention over time-frequency locations
            b, c, h, w = features.shape
            feat_seq = features.permute(0, 2, 3, 1).reshape(b, h * w, c)  # (batch, H*W, C)
            logits_attn = self.attention_head(feat_seq)

            # Combine both paths
            logits = 0.5 * logits_gem + 0.5 * logits_attn
        else:
            logits = logits_gem

        return logits

    def get_feature_maps(self, x):
        """Return raw feature maps from backbone (useful for visualization/debugging)."""
        return self.backbone(x)


# =============================================================================
# Quick Model Test
# =============================================================================
print(f"🏗️ Building model: {CFG.MODEL_NAME}")
model = BirdCLEFModel(CFG, pretrained=True)
dummy_input = torch.randn(2, 1, CFG.N_MELS, 313)  # batch=2, 1 channel, 128 mels, ~5s
dummy_output = model(dummy_input)
print("✅ Model test passed!")
print(f"   • Input shape : {dummy_input.shape}")
print(f"   • Output shape: {dummy_output.shape}")  # (2, 206)
print(f"   • Output sum (should be ~0): {dummy_output.sum().item():.4f}")
# Parameter statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Model statistics:")
print(f"   • Total parameters    : {total_params:,}")
print(f"   • Trainable parameters: {trainable_params:,}")

del model, dummy_input, dummy_output
gc.collect()

🏗️ Building model: tf_efficientnetv2_s
   Backbone feature dim: 1280
   Feature map shape: torch.Size([1, 1280, 4, 10])
✅ Model test passed!
   • Input shape : torch.Size([2, 1, 128, 313])
   • Output shape: torch.Size([2, 206])
   • Output sum (should be ~0): -4.9770

📊 Model statistics:
   • Total parameters    : 22,357,566
   • Trainable parameters: 22,357,566


38

In [21]:
# =============================================================================
# Cell 6: Alternative Model - SED (Sound Event Detection) Style
# =============================================================================
"""
BirdCLEF 2026 - Cell 6: SED-Style Alternative Architecture

This cell defines an alternative model using a **Sound Event Detection (SED)** approach.

Key characteristics:
- Produces both **frame-level** and **clip-level** predictions
- Uses **class-specific attention** to aggregate frame predictions into clip-level output
- Better suited for detecting birds that sing only in specific time segments within the 5-second clip
- More interpretable: attention weights show "where" the model is listening

This architecture is commonly used in audio tagging and sound event detection tasks.

────────────────────────────────────────────────────────────────────────────────
WHAT IS SOUND EVENT DETECTION (SED)?
────────────────────────────────────────────────────────────────────────────────
In traditional audio classification (Cell 5), we treat the entire 5-second clip
as one unit and predict "which bird species is present in this clip?"

In Sound Event Detection (SED), we instead ask:
"Where exactly in this 5-second clip is the bird singing, and which species is it?"

This is much more powerful when:
- Birds only sing for 1-2 seconds inside the 5-second window
- Multiple birds may be present at different times
- We want to know not just "what", but "when"

────────────────────────────────────────────────────────────────────────────────
KEY IDEA OF THIS ARCHITECTURE
────────────────────────────────────────────────────────────────────────────────
1. The backbone extracts rich features at every time-frequency location.
2. We predict bird species probabilities at EVERY location (frame-level).
3. We learn an attention map that tells us "how important each location is"
   for each species.
4. We compute a weighted average of the frame predictions using the attention
   weights → this becomes our final clip-level prediction.

This is called "attention-weighted pooling" or "class-specific attention".
"""


class BirdCLEFSEDModel(nn.Module):
    """
    Sound Event Detection (SED) style model for bird sound classification.
    
    Architecture:
    - Pretrained CNN backbone (same as main model)
    - Frame-level classifier: 1x1 convolution that outputs per-class predictions at every location
    - Class-specific attention: learns to weight important time-frequency regions per species
    - Clip-level prediction: attention-weighted average of frame-level predictions
    
    Input:  (batch, 1, 128, ~313)  ← 5-second mel spectrogram
         │
         ▼
    Backbone (EfficientNetV2 / ConvNeXt / etc.)
         │
         ▼
    Feature Map: (batch, C, H, W)   ← C = 1280~1792 depending on backbone
         │
         ├─→ Frame Classifier (1x1 Conv) → Frame-level logits: (B, 206, H, W)
         │
         └─→ Attention Network (1x1 Conv + Sigmoid) → Attention weights: (B, 206, H, W)
         │
         ▼
    Attention-weighted aggregation → Clip-level logits: (B, 206)
    
    Advantage over standard classification:
    - Can localize bird vocalizations within the 5-second window
    - More robust when birds only sing briefly inside the clip
    
    ────────────────────────────────────────────────────────────────────────────
    WHY CLASS-SPECIFIC ATTENTION?
    ────────────────────────────────────────────────────────────────────────────
    Different bird species have very different vocalization patterns:
    - Some birds sing short, high-pitched bursts
    - Others produce long, low-frequency calls
    - Some have complex songs with multiple frequency bands
    
    By having SEPARATE attention maps per species, the model can learn:
    "For species #47, focus on the lower frequencies around time step 80-120"
    "For species #132, focus on the high-frequency bursts at the beginning"
    
    This is much more powerful than using the same attention for all species.
    """

    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.cfg = cfg
        # ------------------------------------------------------------------
        # 1. Backbone (same as main model)
        # ------------------------------------------------------------------
        self.backbone = timm.create_model(
            cfg.MODEL_NAME,
            pretrained=pretrained,
            in_chans=cfg.IN_CHANNELS,
            num_classes=0,  # Remove original classifier
            global_pool='',  # We will do our own pooling
        )

        # Dynamically get feature dimension
        with torch.no_grad():
            dummy = torch.randn(1, cfg.IN_CHANNELS, cfg.N_MELS, 313)
            features = self.backbone(dummy)
            self.feature_dim = features.shape[1]

        # ------------------------------------------------------------------
        # 2. Frame-level classifier (per location, per class)
        # ------------------------------------------------------------------
        # A 1x1 convolution that predicts "which bird species is active"
        # at every single time-frequency location in the feature map.
        #
        # Output: (batch, 206, H, W) → 206 = number of bird species
        self.frame_classifier = nn.Sequential(
            nn.Conv2d(self.feature_dim, cfg.NUM_CLASSES, kernel_size=1),
        )

        # ------------------------------------------------------------------
        # 3. Class-specific attention (learns importance per species per location)
        # ------------------------------------------------------------------
        # Another 1x1 convolution that outputs an attention weight between 0 and 1
        # for every location and every species.
        #
        # This tells the model "how much should I trust the frame prediction
        # at this location for this particular species?"
        self.attention = nn.Sequential(
            nn.Conv2d(self.feature_dim, cfg.NUM_CLASSES, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass with attention-weighted aggregation.
        
        Args:
            x: Mel spectrogram of shape (batch, 1, n_mels, time_steps)
        Returns:
            clip_logits: Final clip-level predictions of shape (batch, num_classes)
        """
        # Extract rich features from backbone
        features = self.backbone(x)  # Shape: (B, C, H, W)

        # ---------------------------------------------------------------------
        # Step 1: Frame-level predictions
        # ---------------------------------------------------------------------
        # At every location in the spectrogram, predict which bird species
        # is singing. This gives us a "heatmap" of predictions.
        frame_logits = self.frame_classifier(features)  # Shape: (B, 206, H, W)

        # ---------------------------------------------------------------------
        # Step 2: Class-specific attention weights
        # ---------------------------------------------------------------------
        # Learn how important each location is for each species.
        # High attention = "this location is very relevant for this bird"
        attention_weights = self.attention(features)  # Shape: (B, 206, H, W)

        # ---------------------------------------------------------------------
        # Step 3: Attention-weighted aggregation (the key step)
        # ---------------------------------------------------------------------
        # Instead of simple average or max pooling, we do a weighted average:
        #
        # clip_score = Σ (frame_prediction × attention_weight) / Σ (attention_weight)
        #
        # This is mathematically equivalent to:
        # "Take the average of all frame predictions, but give more weight
        #  to locations where the attention is high for that species."
        #
        # The +1e-8 is for numerical stability (avoid division by zero).
        clip_logits = (frame_logits * attention_weights).sum(dim=(-2, -1)) / \
                      (attention_weights.sum(dim=(-2, -1)) + 1e-8)

        return clip_logits


print("✅ SED-style model architecture defined successfully.")
print(f"   This model produces clip-level predictions using class-specific attention.")

✅ SED-style model architecture defined successfully.
   This model produces clip-level predictions using class-specific attention.


In [22]:
# =============================================================================
# Cell 7: Loss Functions
# =============================================================================
"""
BirdCLEF 2026 - Cell 7: Custom Loss Functions

This cell defines specialized loss functions to handle two major challenges in BirdCLEF:

1. **Severe class imbalance** → Many bird species have very few recordings
2. **MixUp augmentation**   → Requires loss functions that support soft labels

Loss functions included:
- FocalLoss        → Focuses training on hard examples (down-weights easy ones)
- BCEFocalLoss     → Combines standard BCE with Focal Loss + class weights
- MixupBCELoss     → BCE loss compatible with soft (fractional) labels from MixUp

These losses are critical for training on highly imbalanced audio datasets.
"""


# =============================================================================
# 1. Focal Loss
# =============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.
    
    Original paper: "Focal Loss for Dense Object Detection" (Lin et al., 2017)
    
    Key idea:
    - Down-weights the contribution of easy examples
    - Focuses training on hard, misclassified examples
    - Particularly effective when one class dominates (e.g., common vs rare birds)
    
    The modulating factor (1 - pt)^gamma reduces the loss for well-classified examples.
    """

    def __init__(self, alpha: float = 1.0, gamma: float = 2.0, reduction: str = 'mean'):
        """
        Args:
            alpha: Balancing parameter (usually set to inverse class frequency)
            gamma: Focusing parameter (higher = more focus on hard examples)
            reduction: 'mean', 'sum', or 'none'
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Raw logits of shape (batch, num_classes)
            targets: Ground truth labels (0 or 1) of shape (batch, num_classes)
        Returns:
            Scalar loss value
        """
        # Binary cross-entropy without reduction
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        # pt = probability of the true class
        pt = torch.exp(-bce_loss)

        # Focal term: (1 - pt)^gamma * BCE
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# =============================================================================
# 2. Combined BCE + Focal Loss (with class weights)
# =============================================================================
class BCEFocalLoss(nn.Module):
    """
    Hybrid loss combining Binary Cross-Entropy and Focal Loss.
    
    Strategy:
    - Use standard BCE with class weights to handle imbalance
    - Add Focal Loss component to focus on hard examples
    - Weighted combination controlled by `focal_weight`
    
    This is often more stable than pure Focal Loss while still benefiting from
    its focusing effect.
    """

    def __init__(self, class_weights: torch.Tensor = None, focal_weight=0.5, gamma: float = 2.0):
        """
        Args:
            class_weights: Tensor of shape (num_classes,) with inverse frequency weights
            focal_weight: Weight given to Focal Loss (0.0 = pure BCE, 1.0 = pure Focal)
            gamma: Focusing parameter for Focal Loss
        """
        super().__init__()
        self.focal = FocalLoss(gamma=gamma)
        self.focal_weight = focal_weight
        self.class_weights = class_weights

    def forward(self, inputs, targets: torch.Tensor) -> torch.Tensor:
        # BCE
        if self.class_weights is not None: # Weighted BCE
            weight = self.class_weights.unsqueeze(0).expand_as(targets)
            bce = F.binary_cross_entropy_with_logits(inputs, targets, weight=weight)
        else: # Standard BCE
            bce = F.binary_cross_entropy_with_logits(inputs, targets)

        # Focal Loss component
        focal = self.focal(inputs, targets)
        
        # Combined loss
        return (1 - self.focal_weight) * bce + self.focal_weight * focal

# =============================================================================
# 3. MixUp-compatible BCE Loss
# =============================================================================
class MixupBCELoss(nn.Module):
    """
    Binary Cross-Entropy loss that supports soft labels (from MixUp / CutMix).
    
    Standard BCE expects targets to be exactly 0 or 1.
    MixUp creates fractional targets (e.g., 0.7 * class_A + 0.3 * class_B).
    
    This loss works directly with soft targets without modification.
    """

    def __init__(self, class_weights: torch.Tensor=None):
        super().__init__()
        self.class_weights = class_weights

    def forward(self, inputs, targets: torch.Tensor) -> torch.Tensor:
        if self.class_weights is not None:
            weight = self.class_weights.unsqueeze(0).expand_as(targets)
            return F.binary_cross_entropy_with_logits(inputs, targets, weight=weight)
        return F.binary_cross_entropy_with_logits(inputs, targets)


print("✅ Loss functions defined.")

✅ Loss functions defined.


In [ ]:
# =============================================================================
# Cell 8: Mixup Implementation
# =============================================================================

def mixup_data(x, y, alpha=0.5):
    """Apply mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]
    
    return mixed_x, mixed_y


def cutmix_data(x, y, alpha=1.0):
    """Apply CutMix augmentation on spectrograms."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    _, _, h, w = x.shape
    cut_h = int(h * math.sqrt(1 - lam))
    cut_w = int(w * math.sqrt(1 - lam))
    
    cy = np.random.randint(h)
    cx = np.random.randint(w)
    
    y1 = np.clip(cy - cut_h // 2, 0, h)
    y2 = np.clip(cy + cut_h // 2, 0, h)
    x1 = np.clip(cx - cut_w // 2, 0, w)
    x2 = np.clip(cx + cut_w // 2, 0, w)
    
    x_cutmix = x.clone()
    x_cutmix[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    
    # Adjust lambda to actual area ratio
    lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)
    mixed_y = lam * y + (1 - lam) * y[index]
    
    return x_cutmix, mixed_y


print("✅ Mixup/CutMix ready.")

In [ ]:
# =============================================================================
# Cell 9: Training Engine
# =============================================================================

class Trainer:
    """Complete training pipeline with best practices."""
    
    def __init__(self, cfg, model, train_loader, val_loader, fold):
        self.cfg = cfg
        self.model = model.to(cfg.DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.fold = fold
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.LR,
            weight_decay=cfg.WEIGHT_DECAY,
        )
        
        # Scheduler
        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=cfg.LR,
            epochs=cfg.EPOCHS,
            steps_per_epoch=len(train_loader),
            pct_start=cfg.WARMUP_EPOCHS / cfg.EPOCHS,
            anneal_strategy='cos',
            final_div_factor=cfg.LR / cfg.MIN_LR,
        )
        
        # Loss
        self.criterion = BCEFocalLoss(
            class_weights=class_weights_tensor,
            focal_weight=0.5,
            gamma=2.0
        )
        
        # Mixed precision
        self.scaler = GradScaler(enabled=cfg.USE_AMP)
        
        # Tracking
        self.best_score = 0
        self.best_loss = float('inf')
        self.history = {
            'train_loss': [], 'val_loss': [],
            'val_f1': [], 'val_map': [], 'lr': []
        }
    
    def train_epoch(self, epoch):
        self.model.train()
        running_loss = 0
        num_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"[Fold {self.fold}] Epoch {epoch+1}")
        
        for batch_idx, (specs, labels) in enumerate(pbar):
            specs = specs.to(self.cfg.DEVICE, non_blocking=True)
            labels = labels.to(self.cfg.DEVICE, non_blocking=True)
            
            # Apply Mixup/CutMix
            if np.random.random() < self.cfg.MIXUP_PROB:
                if np.random.random() < 0.5:
                    specs, labels = mixup_data(specs, labels, self.cfg.MIXUP_ALPHA)
                else:
                    specs, labels = cutmix_data(specs, labels, self.cfg.CUTMIX_ALPHA)
            
            # Forward pass with AMP
            with autocast(enabled=self.cfg.USE_AMP):
                logits = self.model(specs)
                loss = self.criterion(logits, labels)
            
            # Backward pass
            self.optimizer.zero_grad(set_to_none=True)
            self.scaler.scale(loss).backward()
            
            # Gradient clipping
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.scheduler.step()
            
            running_loss += loss.item()
            num_batches += 1
            
            pbar.set_postfix({
                'loss': f'{running_loss/num_batches:.4f}',
                'lr': f'{self.scheduler.get_last_lr()[0]:.2e}'
            })
        
        return running_loss / num_batches
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        running_loss = 0
        all_preds = []
        all_labels = []
        
        for specs, labels in tqdm(self.val_loader, desc="Validating"):
            specs = specs.to(self.cfg.DEVICE, non_blocking=True)
            labels = labels.to(self.cfg.DEVICE, non_blocking=True)
            
            with autocast(enabled=self.cfg.USE_AMP):
                logits = self.model(specs)
                loss = self.criterion(logits, labels)
            
            running_loss += loss.item()
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
        
        avg_loss = running_loss / len(self.val_loader)
        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        
        # Metrics
        # Mean Average Precision (macro)
        try:
            map_score = average_precision_score(all_labels, all_preds, average='macro')
        except:
            map_score = 0.0
        
        # Macro F1 (threshold = 0.5)
        pred_binary = (all_preds > 0.5).astype(int)
        label_binary = all_labels.astype(int)
        try:
            f1 = f1_score(label_binary, pred_binary, average='macro', zero_division=0)
        except:
            f1 = 0.0
        
        return avg_loss, map_score, f1, all_preds, all_labels
    
    def fit(self):
        print(f"\n{'='*60}")
        print(f"🚀 Training Fold {self.fold}")
        print(f"{'='*60}")
        
        for epoch in range(self.cfg.EPOCHS):
            # Train
            train_loss = self.train_epoch(epoch)
            
            # Validate
            val_loss, val_map, val_f1, _, _ = self.validate()
            
            # Track history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_map'].append(val_map)
            self.history['val_f1'].append(val_f1)
            self.history['lr'].append(self.scheduler.get_last_lr()[0])
            
            print(f"\n📊 Epoch {epoch+1}/{self.cfg.EPOCHS}")
            print(f"   Train Loss: {train_loss:.4f}")
            print(f"   Val Loss:   {val_loss:.4f}")
            print(f"   Val mAP:    {val_map:.4f}")
            print(f"   Val F1:     {val_f1:.4f}")
            
            # Save best model
            score = val_map  # Use mAP as primary metric
            if score > self.best_score:
                self.best_score = score
                self.best_loss = val_loss
                save_path = self.cfg.OUTPUT_DIR / f"best_model_fold{self.fold}.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'best_score': self.best_score,
                    'config': {k: v for k, v in vars(self.cfg).items() 
                              if not k.startswith('_')},
                }, save_path)
                print(f"   ✅ New best model saved! mAP: {self.best_score:.4f}")
        
        return self.history
    
    def plot_history(self):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        axes[0].plot(self.history['train_loss'], label='Train', linewidth=2)
        axes[0].plot(self.history['val_loss'], label='Val', linewidth=2)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title(f'Loss (Fold {self.fold})', fontweight='bold')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(self.history['val_map'], label='mAP', linewidth=2, color='green')
        axes[1].plot(self.history['val_f1'], label='F1', linewidth=2, color='orange')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Score')
        axes[1].set_title(f'Validation Metrics (Fold {self.fold})', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        axes[2].plot(self.history['lr'], linewidth=2, color='purple')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Learning Rate')
        axes[2].set_title('Learning Rate Schedule', fontweight='bold')
        axes[2].set_yscale('log')
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'training_history_fold{self.fold}.png', dpi=150, bbox_inches='tight')
        plt.show()


print("✅ Training engine ready.")

In [ ]:
# =============================================================================
# Cell 10: Run Training
# =============================================================================

all_histories = {}

for fold in CFG.TRAIN_FOLDS:
    # Data splits
    train_df = df_filtered[df_filtered['fold'] != fold].reset_index(drop=True)
    val_df = df_filtered[df_filtered['fold'] == fold].reset_index(drop=True)
    
    print(f"\n📁 Fold {fold}: Train={len(train_df)}, Val={len(val_df)}")
    
    # Datasets
    train_dataset = BirdCLEFDataset(train_df, CFG, mode='train')
    val_dataset = BirdCLEFDataset(val_df, CFG, mode='val')
    
    # Weighted sampler for class imbalance
    sample_weights = class_weights[train_df['label_idx'].values]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_dataset),
        replacement=True
    )
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.BATCH_SIZE,
        sampler=sampler,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY,
        drop_last=True,
        persistent_workers=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG.BATCH_SIZE * 2,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=CFG.PIN_MEMORY,
        persistent_workers=True,
    )
    
    # Model
    model = BirdCLEFModel(CFG, pretrained=CFG.PRETRAINED)
    
    # Train
    trainer = Trainer(CFG, model, train_loader, val_loader, fold)
    history = trainer.fit()
    trainer.plot_history()
    
    all_histories[fold] = history
    
    # Cleanup
    del model, trainer, train_loader, val_loader, train_dataset, val_dataset
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"\n{'='*60}")
    print(f"✅ Fold {fold} complete. Best mAP: {history['val_map'][-1]:.4f}")
    print(f"{'='*60}")